# Karlsson dark segments — visualization and correlation of latent variables 

Workflow:

1. Use cells from previous Karlsson analysis notebooks to generate data files (`karlsson_dark_segments_128.npz`) and `karlsson_dark_segments_table.pkl`).
2. Normalize each segment by its own maximum, exactly as A7X expects.
3. Load the pretrained A7X-8 model.
4. Encode the **entire** darkness dataset at once. 
5. Build a single 3-part figure:
   - **A.** Distributions of the 8 latents across the dataset (1×8 panels).
   - **B.** 8×8 correlation matrix between latent pairs.
   - **C.** A7X reconstructions vs. true original curves for 16 randomly sampled segments (2×8 panels).


In [ ]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 0. Generate needed data files, containing dark segment A7X representations.  

With this code, the dark trajectories will be split into non-overlapping 128-point
segments (points 0–127, 128–255, 256–383 of each 385-point trajectory) by the
earlier preprocessing notebook. We simply load those saved outputs here:

- `karlsson_dark_segments_128.npz` — flattened `(N, 128)` segment array plus IDs.
- `karlsson_dark_segments_table.pkl` — segment-level metadata table.

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# Preferred location of the master pickle
PKL_PATH = Path("analyses/karlsson/outputs/karlsson_master.pkl")

# If the preferred path does not exist, search locally for the file.
if not PKL_PATH.exists():
    print(f"Preferred PKL file not found at: {PKL_PATH.resolve()}")
    print("Searching for karlsson_master.pkl in the current directory...")

    local_hits = []  # Avoid silently loading stale data from another directory.

    if local_hits:
        PKL_PATH = local_hits[0]
        print(f"Found master PKL at: {PKL_PATH.resolve()}")
    else:
        raise FileNotFoundError(
            "\nCould not find karlsson_master.pkl.\n\n"
            "Please run Karlsson_dark_preprocessing.ipynb first.\n"
            "The builder notebook should create:\n"
            "    analyses/karlsson/outputs/karlsson_master.pkl\n\n"
            "Also check that you are running this notebook from the "
            "correct project directory."
        )

# ---------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------

OUTPUT_DIR = PKL_PATH.parent / "dark_segment_a7x_pca_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading master pkl from: {PKL_PATH.resolve()}")
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")

# ---------------------------------------------------------------------
# Load master pickle
# ---------------------------------------------------------------------

try:
    with open(PKL_PATH, "rb") as f:
        master = pickle.load(f)
except Exception as e:
    raise RuntimeError(
        f"\nThe file was found, but could not be loaded:\n"
        f"    {PKL_PATH.resolve()}\n\n"
        f"Original error: {type(e).__name__}: {e}"
    ) from e

# ---------------------------------------------------------------------
# Validate master structure
# ---------------------------------------------------------------------

if not isinstance(master, dict):
    raise TypeError(
        f"\nExpected the pickle to contain a dictionary, "
        f"but found: {type(master).__name__}"
    )

if "tables" not in master:
    raise KeyError(
        "\nThe loaded master pickle does not contain the expected "
        "'tables' key.\n"
        f"Available top-level keys: {list(master.keys())}"
    )

print("Top-level keys:", list(master.keys()))
print("Tables:", list(master["tables"].keys()))


In [ ]:
traj = master["tables"]["trajectories"].copy()

# Ensure trajectory_id is present as a column.
if "trajectory_id" not in traj.columns:
    if traj.index.name == "trajectory_id":
        traj = traj.reset_index()
    else:
        traj["trajectory_id"] = traj.index.astype(str)
traj["trajectory_id"] = traj["trajectory_id"].astype(str)

# Use the index built by the master notebook when possible.
try:
    ids_dark = master["indexes"]["by_treatment"][("darkness",)]
    dark = traj[traj["trajectory_id"].isin(ids_dark)].copy()
except Exception:
    dark = traj[traj["treatment"].astype(str).eq("darkness")].copy()

# Reproducible order.
sort_cols = [c for c in ["organism", "species_code", "replicate", "plate_type", "well", "trajectory_id"] if c in dark.columns]
if sort_cols:
    dark = dark.sort_values(sort_cols).reset_index(drop=True)
else:
    dark = dark.sort_values("trajectory_id").reset_index(drop=True)

print("Dark trajectories:", dark.shape)
print("Columns:", list(dark.columns))

# Check point counts.
dark["n_points_original"] = dark["values"].apply(lambda x: len(np.asarray(x)))
print(dark["n_points_original"].describe())
print("Unique point counts:", sorted(dark["n_points_original"].unique())[:20])

display(dark.head())

In [ ]:
SEGMENT_LENGTH = 128
N_SEGMENTS_PER_TRAJECTORY = 3
REQUIRED_POINTS = SEGMENT_LENGTH * N_SEGMENTS_PER_TRAJECTORY  # 384

segment_rows = []
segment_values = []
segment_times = []
skipped = []

for _, row in dark.iterrows():
    values = np.asarray(row["values"], dtype=np.float32)
    times = np.asarray(row.get("time_hours", np.arange(len(values))), dtype=np.float32)
    if len(values) < REQUIRED_POINTS:
        skipped.append((row["trajectory_id"], len(values)))
        continue

    # Drop any points beyond the first 384, including point 385 when present.
    values_384 = values[:REQUIRED_POINTS]
    times_384 = times[:REQUIRED_POINTS]

    for seg_idx in range(N_SEGMENTS_PER_TRAJECTORY):
        start = seg_idx * SEGMENT_LENGTH
        end = start + SEGMENT_LENGTH
        seg_values = values_384[start:end]
        seg_times = times_384[start:end]

        meta = row.drop(labels=["values", "time_hours"], errors="ignore").to_dict()
        source_id = str(row["trajectory_id"])
        meta.update({
            "source_trajectory_id": source_id,
            "segment_id": f"{source_id}__seg{seg_idx + 1}",
            "segment_index": seg_idx + 1,
            "segment_start_idx_0based": start,
            "segment_end_idx_exclusive_0based": end,
            "segment_length": SEGMENT_LENGTH,
            "dropped_after_idx_0based": REQUIRED_POINTS,
            "n_points_used_from_original": REQUIRED_POINTS,
            "n_points_original": len(values),
            "segment_time_start": float(seg_times[0]) if len(seg_times) else np.nan,
            "segment_time_end": float(seg_times[-1]) if len(seg_times) else np.nan,
            "segment_min": float(np.nanmin(seg_values)),
            "segment_max": float(np.nanmax(seg_values)),
            "segment_mean": float(np.nanmean(seg_values)),
            "segment_auc_index_space": float(np.trapz(seg_values)),
        })
        segment_rows.append(meta)
        segment_values.append(seg_values)
        segment_times.append(seg_times)

segments_table = pd.DataFrame(segment_rows)
X_segments = np.vstack(segment_values).astype(np.float32)
T_segments = np.vstack(segment_times).astype(np.float32)

print("Segments table:", segments_table.shape)
print("Flattened segment array:", X_segments.shape)
print("Segment time array:", T_segments.shape)
print("Skipped trajectories:", len(skipped))
if skipped:
    print("First skipped examples:", skipped[:10])

assert X_segments.shape[0] == segments_table.shape[0]
assert X_segments.shape[1] == SEGMENT_LENGTH
assert not np.isnan(X_segments).any(), "NaN values detected in segment matrix."

display(segments_table.head())

In [ ]:
# Save the segment labels / metadata table.
segments_table_path_pkl = OUTPUT_DIR / "karlsson_dark_segments_table.pkl"
segments_table_path_csv = OUTPUT_DIR / "karlsson_dark_segments_table.csv.gz"
segments_table.to_pickle(segments_table_path_pkl)
segments_table.to_csv(segments_table_path_csv, index=False)

# Save the arrays.
npz_path = OUTPUT_DIR / "karlsson_dark_segments_128.npz"
np.savez_compressed(
    npz_path,
    X_segments=X_segments,
    T_segments=T_segments,
    segment_id=segments_table["segment_id"].astype(str).to_numpy(),
    source_trajectory_id=segments_table["source_trajectory_id"].astype(str).to_numpy(),
    segment_index=segments_table["segment_index"].to_numpy(),
)

print("Saved:")
print(" -", segments_table_path_pkl)
print(" -", segments_table_path_csv)
print(" -", npz_path)

## 1. Load the preprocessed dark segments.

In [ ]:
CANDIDATE_OUTPUT_DIRS = [Path("analyses/karlsson/outputs/dark_segment_a7x_pca_outputs")]

OUTPUT_DIR = next((p for p in CANDIDATE_OUTPUT_DIRS if (p / "karlsson_dark_segments_128.npz").exists()), None)

if OUTPUT_DIR is None:
    local_hits = []  # Avoid silently loading stale segment files.
    if local_hits:
        OUTPUT_DIR = local_hits[0].parent

if OUTPUT_DIR is None:
    raise FileNotFoundError(
        "Could not find karlsson_dark_segments_128.npz. "
        "Run the preprocessing/segmentation notebook first to generate the "
        "128-point dark segments used here."
    )

print("Using preprocessed segments from:", OUTPUT_DIR.resolve())

seg_npz = np.load(OUTPUT_DIR / "karlsson_dark_segments_128.npz", allow_pickle=True)
X_segments = seg_npz["X_segments"].astype(np.float32)  # (N, 128) raw curves
T_segments = seg_npz["T_segments"].astype(np.float32)  # (N, 128) real time (hours)
segment_id = seg_npz["segment_id"]
source_trajectory_id = seg_npz["source_trajectory_id"]
segment_index = seg_npz["segment_index"]

segments_table = pd.read_pickle(OUTPUT_DIR / "karlsson_dark_segments_table.pkl")

print("X_segments:", X_segments.shape)
print("segments_table:", segments_table.shape)
assert X_segments.shape[0] == segments_table.shape[0]
assert X_segments.shape[1] == 128

## 2. Normalize segments for A7X

Each segment is normalized by its own maximum before A7X encoding, following the
existing convention. The raw per-segment maximum is kept so reconstructions can be
mapped back to original curve scale.

In [ ]:
def prepare_curves_for_a7x(curves):
    """Normalize curves by per-curve maximum and return tensor plus raw maxima."""
    curves = np.asarray(curves, dtype=np.float32)
    if curves.ndim == 1:
        curves = curves[None, :]
    if curves.shape[1] != 128:
        raise ValueError(f"Expected curves with shape (N, 128), got {curves.shape}")

    raw_max = np.nanmax(curves, axis=1, keepdims=True).astype(np.float32)
    denom = np.where(raw_max > 0, raw_max, 1.0).astype(np.float32)
    curves_norm = curves / denom
    x = torch.from_numpy(curves_norm).unsqueeze(1)  # (N, 1, 128)
    return x, raw_max, curves_norm.astype(np.float32)

X_tensor, raw_max, X_segments_norm = prepare_curves_for_a7x(X_segments)
print("X_segments_norm:", X_segments_norm.shape)
print("raw_max:", raw_max.shape)

## 3. Load pretrained A7X-8

Requirements on the machine running this notebook:

- The repository source code should be available at `./src`, so that
  `ml.models.autoencoder7x` can be imported.
- Internet or cached Hugging Face files are needed for `you-lab/autoencoder7x-08`
  unless the model is already cached.

In [ ]:
from huggingface_hub import hf_hub_download

repo_dir = Path("./")
sys.path.append(str(repo_dir / "src"))

try:
    from ml.models.autoencoder7x import Autoencoder7XModel
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Could not import Autoencoder7XModel from ml.models.autoencoder7x. "
        "Run this notebook from the project root where ./src is available, or update repo_dir above."
    ) from e

REPO_ID = "you-lab/autoencoder7x-08"


def load_a7x_model(repo_id=REPO_ID, device=DEVICE):
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
    weights_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")

    with open(config_path, "r") as f:
        config = json.load(f)
    config.pop("model_type", None)

    model = Autoencoder7XModel(config)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

fm = load_a7x_model()
fm.eval()
print(fm.__class__.__name__, "loaded")

## 4. Compress the entire dataset with A7X

Every dark segment is encoded to
obtain its 8-dimensional latent `mu`, `log_var`, and normalized reconstruction.

In [ ]:
def a7x_forward_with_mu_logvar(model, xb):
    """Return reconstruction, mu, and log_var from an A7X forward pass."""
    out = model(xb)

    if isinstance(out, dict):
        recon = out.get("recon", out.get("reconstruction", out.get("x_recon", None)))
        mu = out.get("mu", out.get("z", out.get("latent", None)))
        log_var = out.get("log_var", out.get("logvar", None))
        if recon is not None and mu is not None and log_var is not None:
            return recon, mu, log_var

    if isinstance(out, (tuple, list)):
        if len(out) >= 3:
            recon, mu, log_var = out[0], out[1], out[2]
            return recon, mu, log_var
        if len(out) == 2:
            raise RuntimeError(
                "A7X returned only two outputs. Need log_var for this analysis."
            )

    raise RuntimeError(
        "Unexpected A7X forward output. Inspect model.forward output and update "
        "a7x_forward_with_mu_logvar."
    )


def encode_segments_a7x(curves, model, batch_size=1024, device=DEVICE):
    x, raw_max, curves_norm = prepare_curves_for_a7x(curves)
    mu_list, logvar_list, recon_list = [], [], []

    model.eval()
    with torch.no_grad():
        for start in range(0, x.shape[0], batch_size):
            xb = x[start:start + batch_size].to(device)
            recon, mu, log_var = a7x_forward_with_mu_logvar(model, xb)
            recon_list.append(recon.detach().cpu().view(recon.shape[0], -1).numpy())
            mu_list.append(mu.detach().cpu().view(mu.shape[0], -1).numpy())
            logvar_list.append(log_var.detach().cpu().view(log_var.shape[0], -1).numpy())

    recon_norm = np.vstack(recon_list).astype(np.float32)
    mu = np.vstack(mu_list).astype(np.float32)
    log_var = np.vstack(logvar_list).astype(np.float32)
    return mu, log_var, recon_norm, raw_max, curves_norm


mu_a7x, logvar_a7x, recon_a7x_norm, raw_max, X_segments_norm = encode_segments_a7x(
    X_segments, fm, batch_size=1024, device=DEVICE,
)

# Reconstructions mapped back to original curve scale.
recon_a7x_raw = recon_a7x_norm * raw_max

print("mu_a7x:", mu_a7x.shape)
print("logvar_a7x:", logvar_a7x.shape)
print("recon_a7x_norm:", recon_a7x_norm.shape)
assert mu_a7x.shape[1] == 8, "Expected A7X-8 to produce 8 latent dimensions."

In [ ]:
latent_npz_path = OUTPUT_DIR / "karlsson_dark_segments_a7x_latents_reconstruction.npz"
np.savez_compressed(
    latent_npz_path,
    mu=mu_a7x,
    log_var=logvar_a7x,
    recon_norm=recon_a7x_norm,
    raw_max=raw_max,
    X_segments_norm=X_segments_norm,
    segment_id=segment_id,
    source_trajectory_id=source_trajectory_id,
    segment_index=segment_index,
)
print("Saved:", latent_npz_path)

## 5. Summary figure

- **A.** Distribution of each of the 8 A7X latents (`mu`) across the dataset.
- **B.** 8×8 correlation matrix between latent pairs.
- **C.** A7X reconstructions vs. true original curves for 16 randomly sampled segments.

In [ ]:
n_latents = mu_a7x.shape[1]
corr = np.corrcoef(mu_a7x.T)

n_examples = 16
example_idx = rng.choice(X_segments.shape[0], size=min(n_examples, X_segments.shape[0]), replace=False)

FONT_SIZE = 16  
plt.rcParams.update({"font.size": FONT_SIZE})

fig = plt.figure(figsize=(20, 20))
outer = gridspec.GridSpec(3, 1, height_ratios=[1.0, 3.2, 2.2], hspace=0.55)

# --- Panel A: 1x8 latent distributions -------------------------------
gsA = gridspec.GridSpecFromSubplotSpec(1, n_latents, subplot_spec=outer[0], wspace=0.38)
for j in range(n_latents):
    ax = fig.add_subplot(gsA[0, j])
    ax.hist(mu_a7x[:, j], bins=40, color="#4C72B0")
    ax.set_title(rf"$\mu_{{{j+1}}}$", fontsize=FONT_SIZE + 2, fontweight="bold")
    if j == 0:
        ax.set_ylabel("Count", fontsize=FONT_SIZE)
    ax.tick_params(labelsize=FONT_SIZE - 2)

# --- Panel B: 8x8 latent correlation matrix ---------------------------
gsB = gridspec.GridSpecFromSubplotSpec(1, 1, subplot_spec=outer[1])
axB = fig.add_subplot(gsB[0, 0])
axB.set_anchor("W")
im = axB.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")

mu_labels = [rf"$\mu_{{{i+1}}}$" for i in range(n_latents)]
axB.set_xticks(range(n_latents))
axB.set_xticklabels(mu_labels, fontsize=FONT_SIZE)
axB.set_yticks(range(n_latents))
axB.set_yticklabels(mu_labels, fontsize=FONT_SIZE)
cax = axB.inset_axes([1.1, 0.0, 0.04, 1.0])  # [x0, y0, width, height] in axB-fraction units
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Pearson correlation", fontsize=FONT_SIZE)
cbar.ax.tick_params(labelsize=FONT_SIZE - 2)

# --- Panel C: 2x8 reconstruction vs. true curves -----------------------
gsC = gridspec.GridSpecFromSubplotSpec(2, 8, subplot_spec=outer[2], wspace=0.35, hspace=0.45)
line_handles = None
for k, idx in enumerate(example_idx):
    row, col = divmod(k, 8)
    ax = fig.add_subplot(gsC[row, col])
    h_true, = ax.plot(X_segments[idx], color="k", lw=1.5, label="True")
    h_recon, = ax.plot(recon_a7x_raw[idx], "--", color="#DD8452", lw=1.5, label="Reconstructed")
    ax.tick_params(labelsize=FONT_SIZE - 4)
    if k == 0:
        ax.set_xlabel("Time index", fontsize=FONT_SIZE - 2)
        ax.set_ylabel("Value", fontsize=FONT_SIZE - 2)
        line_handles = [h_true, h_recon]

# shared legend placed above panel C, outside all individual axes
posC_full = gsC[0, 0].get_position(fig) 
fig.legend(handles=line_handles, labels=["True","Reconstructed"],
           loc="upper center", bbox_to_anchor=(0.22, outer[2].get_position(fig).y1 + 0.025),
           ncol=2, fontsize=FONT_SIZE-4, frameon=False)

# --- Panel labels A / B / C --------------------------------------------
posA = gsA[0, 0].get_position(fig)
posB = axB.get_position(fig)
posC = gsC[0, 0].get_position(fig)
fig.text(0.06, posA.y1 + 0.01, "A", fontsize=22, fontweight="bold")
fig.text(0.06, posB.y1 + 0.01, "B", fontsize=22, fontweight="bold")
fig.text(0.06, posC.y1 + 0.01, "C", fontsize=22, fontweight="bold")

fig.savefig(OUTPUT_DIR / "a7x_figr2.pdf", bbox_inches="tight")
plt.show()
print("Saved:", OUTPUT_DIR / "a7x_figr2.pdf")